# Training_File_Floor.ipynb — STEP 2D Floor Diagnostic

**Branch git**: `Floor_Diagnostic` (eredita infra da `Optimizer_Exploration`).

**Contesto**: STEP 2C ha confermato che il plateau val~0.28 NON è risolvibile da:
- capacità (sweep STEP 2B su 11× hidden range)
- ottimizzatore (sweep STEP 2C: Prodigy lr=0.1, AdamW, comparable Δ=0.18%)
- scheduler (OneCycle vs none)

Dato che **8 setup diversi convergono nella banda 0.279-0.282**, il floor è **strutturale**.
Questo notebook identifica QUALE delle 4 cause candidate è responsabile.

**Configurazione baseline (vincente STEP 2C)**: AdamW b=8 + OneCycle + h=32, r=8 + 15 ep × 190 cap.

**Plan in ordine di costo/utilità** (Po2 DIFFERITA per scelta utente):

| Plan | Test | Modifica vs baseline | Costo Azure | Predict colpevole |
|------|------|----------------------|-------------|-------------------|
| **F1** | PINN no-multi-obj | `lambda_phys=lambda_ou=lambda_bc=0` | ~45 min | ~50% (most likely) |
| **F2** | OU noise OFF | `--noise_scale 0.0` (cache diverso) | ~45 min | ~15% |
| **F3** | Dataset 3.3× | `--n_train 5000` (cache nuovo, 3× più grande) | ~2-3h | ~10% |
| ~~F4~~ | ~~Po2 quantization OFF~~ | _differito (esperimenti utente prima)_ | — | ~25% |

**Output atteso**:
- Se F1 batte 0.27 → floor era PINN trade-off → STEP 2E: ribilanciare λ
- Se F2 batte 0.27 → floor era OU noise (irriducibile in produzione, ma sappiamo)
- Se F3 batte 0.27 → floor era data diversity → STEP 2E: dataset più grande/diverso
- Se TUTTI restano a 0.28 → floor è Po2 quantization (test F4 differito)

**Reference**: AdamW b=8 del branch Optimizer_Exploration ha val=0.2805. Lo useremo come baseline.

In [ ]:
# ===========================================================
# CELLA 0 — Bootstrap deps + git checkout Floor_Diagnostic
# ===========================================================
import sys

print('📦 Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas
!{sys.executable} -m pip install --quiet nbstripout

# nbstripout per evitare conflitti git sui notebook (output stripping)
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   ✅ deps installed + nbstripout attivato')

# Sync con remote
print('\n🔄 git fetch + checkout Floor_Diagnostic + pull:')
!git fetch origin
!git checkout Floor_Diagnostic 2>/dev/null || git checkout -b Floor_Diagnostic origin/Floor_Diagnostic
!git pull --no-rebase --no-edit origin Floor_Diagnostic

print('\n📍 Branch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 — ENV CHECK
# ===========================================================
import sys, platform, os

print(f'🐍 Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'📂 Working dir:     {os.getcwd()}')

checks = []
for mod in ['torch', 'numpy', 'pandas', 'matplotlib']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'   ✅ {mod:<14} v{v}')
        checks.append((mod, True))
    except Exception as e:
        print(f'   ❌ {mod:<14} {e}')
        checks.append((mod, False))

import torch
if torch.cuda.is_available():
    print(f'\n🎮 CUDA:            ✅ {torch.cuda.get_device_name(0)}')
else:
    print(f'\n🎮 CUDA:            ❌ training su CPU')

print('\n📁 File critici:')
for f in ['train.py', 'config.py', 'core/network.py', 'data/generator.py']:
    exists = os.path.isfile(f)
    print(f'   {"✅" if exists else "❌"} {f}')
    checks.append((f, exists))

# Verifica nuovo CLI --noise_scale
import subprocess
res = subprocess.run([sys.executable, 'train.py', '--help'], capture_output=True, text=True)
has_noise = 'noise_scale' in (res.stdout + res.stderr)
print(f'\n🔧 train.py supporta --noise_scale: {"✅" if has_noise else "❌ (branch sbagliato? rifare Cella 0)"}')
checks.append(('noise_scale CLI', has_noise))

n_fail = sum(1 for _, ok in checks if not ok)
if n_fail == 0:
    print('\n✅ ENV CHECK PASSED — procedi con Cella 2')
else:
    print(f'\n❌ ENV CHECK FAILED — {n_fail} problemi')
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 — FLOOR_PLAN: 3 config diagnostiche
# ===========================================================
# Configurazione comune: AdamW vincente di STEP 2C (Plan B Optimizer_Exploration).
# Ogni Plan varia UNA SOLA cosa per isolare il colpevole del floor.
#
# ⚠️  ATTENZIONE asimmetria F3: con n_train=5000 e stesso budget step (2850),
#     reps scende a ~0.10 (vs 0.32 di F1/F2). Se F3 non migliora val ma
#     SOSPETTI sia per under-exposure, puoi bumpare max_steps_per_epoch a 400
#     (= 6000 step → reps ~0.20, tempo 3h) — vedi nota sotto.

import time, subprocess, sys, os, datetime, shutil, glob, json
import pandas as pd, numpy as np

FLOOR_COMMON = {
    'n_train':              1500,       # F1, F2; F3 lo overrida
    'n_val':                300,
    'seq_len':              50,
    'scenario_mix':         'highway',
    'cut_in_ratio':         0.0,
    'cf_hidden_size':       32,
    'cf_rank':              8,
    # PINN weights nominali (F1 li overrida a 0)
    'lambda_data':          1.0,
    'lambda_phys':          0.1,
    'lambda_ou':            0.05,
    'lambda_bc':            1.0,
    # OU noise nominale (F2 lo overrida a 0)
    'noise_scale':          1.0,
    # AdamW vincente STEP 2C
    'optimizer':            'adamw',
    'lr':                   2e-3,
    'batch_size':           8,
    'val_batch_size':       64,
    'scheduler':            'onecycle',
    'max_lr':               2e-3,
    'epochs':               15,
    'max_steps_per_epoch':  190,        # 190 × 15 = 2850 step (= STEP 2C Plan B)
    # Safety: no abort, no early stop (vediamo tutto fino in fondo)
    'max_inf_streak':       99999,
    'early_stop_patience':  0,
    'early_stop_delta':     0.005,
}

FLOOR_PLAN = [
    # ── F1: PINN no-multi-obj (solo data loss) ──────────────────────
    # Ipotesi: il trade-off 5-obiettivi limita val. Disattivando phys/ou/bc
    # vediamo se val_data scende sotto 0.27.
    {
        'tag':            'P12_S2D_F1_no_pinn',
        'lambda_phys':    0.0,
        'lambda_ou':      0.0,
        'lambda_bc':      0.0,
        # lambda_data resta 1.0
    },
    # ── F2: OU noise OFF (dataset deterministico) ───────────────────
    # Ipotesi: l'OU noise (NOISE_GAP_REL/VEL_OPT/ACCEL) introduce errore
    # irriducibile. Con noise_scale=0 vediamo il floor 'puro' senza rumore.
    # Cache filename diverso (cache_1500_highway_cut0.0_ou0.0.pt) → fresh gen.
    {
        'tag':            'P12_S2D_F2_no_ou',
        'noise_scale':    0.0,
    },
    # ── F3: Dataset 3.3× (n_train=5000) ─────────────────────────────
    # Ipotesi: 1500 traiettorie non bastano. Con 5000 dataset 3.3× più grande.
    # NOTA: con stesso budget step (2850), reps cala a ~0.10 (vs 0.32 di F1/F2).
    # Se vuoi reps simili (~0.30), aggiungi 'max_steps_per_epoch': 600
    # (= 9000 step, ~3h Azure). Lasciato 190 per ora per non sforare il tempo.
    {
        'tag':            'P12_S2D_F3_dataset_big',
        'n_train':        5000,
    },
]

# Quali eseguire? Modifica per disabilitare singoli plan.
RUN_FLOOR = ['F1', 'F2', 'F3']

def _cache_path(plan):
    """Costruisce path cache. Include _ou<scale> se diverso da 1.0 per evitare
    collisione con cache esistenti generate con noise nominale."""
    full = {**FLOOR_COMMON, **plan}
    base = f'data/cache_{full["n_train"]}_{full["scenario_mix"]}_cut{full["cut_in_ratio"]}'
    if full['noise_scale'] != 1.0:
        base += f'_ou{full["noise_scale"]}'
    return f'{base}.pt'

print(f'🔬 FLOOR DIAGNOSTIC: {len(FLOOR_PLAN)} config')
print(f'⚠️  Po2 quantization NON testata (differita per scelta utente)')
print(f'⚠️  Safety: max_inf_streak=99999, early_stop=0')
print(f'📊 Reference: AdamW b=8 STEP 2C Plan B → val=0.2805 (per confronto)')

for i, plan in enumerate(FLOOR_PLAN, 1):
    label = f'F{i}'
    active = '✅' if label in RUN_FLOOR else '⏸️'
    full = {**FLOOR_COMMON, **plan}
    cp = _cache_path(plan)
    cache_exists = '📦 esistente' if os.path.isfile(cp) else '🆕 da generare'
    n_w_est = full['n_train'] * 47  # ~47 windows/traj
    n_steps = full['max_steps_per_epoch'] * full['epochs']
    reps    = n_steps * full['batch_size'] / n_w_est
    reps_warn = ' ⚠️ low reps!' if reps < 0.15 else ''
    print(f'\n  {active} {label}: {full["tag"]}')
    print(f'     override: {list(plan.keys())[1:] if len(plan) > 1 else "(see lambda)"}')
    print(f'     lambda_data={full["lambda_data"]} phys={full["lambda_phys"]} ou={full["lambda_ou"]} bc={full["lambda_bc"]}')
    print(f'     n_train={full["n_train"]}  noise_scale={full["noise_scale"]}')
    print(f'     cache: {cp}  ({cache_exists})')
    print(f'     budget: {full["max_steps_per_epoch"]}/ep × {full["epochs"]} ep = {n_steps} step  (reps≈{reps:.3f}{reps_warn})')

In [ ]:
# ===========================================================
# CELLA 3 — RUN sweep diagnostic (continue-on-failure)
# ===========================================================
def _build_cli_floor(plan):
    full = {**FLOOR_COMMON, **plan}
    args = [
        sys.executable, 'train.py',
        '--epochs',              str(full['epochs']),
        '--n_train',             str(full['n_train']),
        '--n_val',               str(full['n_val']),
        '--batch_size',          str(full['batch_size']),
        '--val_batch_size',      str(full['val_batch_size']),
        '--seq_len',             str(full['seq_len']),
        '--scheduler',           full['scheduler'],
        '--max_lr',              str(full['max_lr']),
        '--lr',                  str(full['lr']),
        '--optimizer',           full['optimizer'],
        '--scenario_mix',        full['scenario_mix'],
        '--cut_in_ratio',        str(full['cut_in_ratio']),
        '--cf_hidden_size',      str(full['cf_hidden_size']),
        '--cf_rank',             str(full['cf_rank']),
        '--lambda_data',         str(full['lambda_data']),
        '--lambda_phys',         str(full['lambda_phys']),
        '--lambda_ou',           str(full['lambda_ou']),
        '--lambda_bc',           str(full['lambda_bc']),
        '--noise_scale',         str(full['noise_scale']),
        '--max_inf_streak',      str(full['max_inf_streak']),
        '--early_stop_patience', str(full['early_stop_patience']),
        '--early_stop_delta',    str(full['early_stop_delta']),
        '--max_steps_per_epoch', str(full['max_steps_per_epoch']),
        '--data_cache',          _cache_path(plan),
        '--tag',                 full['tag'],
    ]
    return args

def _push_floor_result(plan):
    full = {**FLOOR_COMMON, **plan}
    tag = full['tag']
    src, dst = f'checkpoints/{tag}', f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   ⚠️  {src} mancante')
        return False
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str, status = '', 'unknown'
    if os.path.isfile(f'{dst}/training_log.csv'):
        edf = pd.read_csv(f'{dst}/training_log.csv')
        if len(edf) > 0:
            val_str = f'best val={edf.val_total.min():.4f} (E{int(edf.val_total.idxmin())+1}/{len(edf)})'
            status = 'FROZEN' if (len(edf) > 2 and edf.val_total.iloc[1:].nunique() == 1) else 'OK'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (S2D Floor): {tag} ({ts})\n\n'
        f'Status: {status}  |  {val_str}\n'
        f'Override: lambda_phys={full["lambda_phys"]} ou={full["lambda_ou"]} bc={full["lambda_bc"]}, '
        f'noise_scale={full["noise_scale"]}, n_train={full["n_train"]}\n'
        f'Cache: {_cache_path(plan)}\n'
        f'Branch Floor_Diagnostic\n'
    )
    with open('/tmp/floor_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst}
    !git commit -F /tmp/floor_msg.txt
    !rm /tmp/floor_msg.txt
    !git pull --no-rebase --no-edit origin Floor_Diagnostic
    !git push origin Floor_Diagnostic
    return True

# ── Esecuzione ─────────────────────────────────────────────────
floor_results = []
t_start = time.time()

for i, plan in enumerate(FLOOR_PLAN, 1):
    label = f'F{i}'
    if label not in RUN_FLOOR:
        print(f'\n⏸️  Skipping {label} ({plan["tag"]})')
        continue
    full = {**FLOOR_COMMON, **plan}
    print('\n' + '=' * 78)
    print(f'🔬 {label}: {full["tag"]}')
    print('=' * 78)
    cmd = _build_cli_floor(plan)
    print(f'CLI: {" ".join(cmd[2:])}\n')
    t0 = time.time()
    res = subprocess.run(cmd, capture_output=False)
    elapsed = time.time() - t0
    status = 'ok' if res.returncode == 0 else f'fail (exit {res.returncode})'
    print(f'\n⏱️  {label} terminato in {elapsed/60:.1f}min — exit: {status}')
    print(f'\n📤 Push results/{full["tag"]}...')
    _push_floor_result(plan)
    floor_results.append({'label': label, 'tag': full['tag'], 'status': status, 'elapsed': elapsed})

print(f'\n{"="*78}\n🏁 FLOOR SWEEP COMPLETATO in {(time.time()-t_start)/60:.1f} min\n{"="*78}')
for r in floor_results:
    print(f"   {r['label']:<4} {r['tag']:<32} {r['status']:<20} {r['elapsed']/60:.1f}min")

In [ ]:
# ===========================================================
# CELLA 4 — ANALISI: confronto F1/F2/F3 vs baseline AdamW
# ===========================================================
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

BASELINE_TAG = 'P12_S2C_planB_adamw_b8'   # da branch Optimizer_Exploration
BASELINE_VAL = 0.2805

# Carica risultati floor
loaded = []
for plan in FLOOR_PLAN:
    full = {**FLOOR_COMMON, **plan}
    tag = full['tag']
    base = f'results/{tag}'
    if not os.path.isdir(base):
        print(f'⏭️  {tag}: no results, skip')
        continue
    ep_csv = f'{base}/training_log.csv'
    if not os.path.isfile(ep_csv):
        continue
    ep = pd.read_csv(ep_csv)
    loaded.append({'plan': plan, 'full': full, 'ep': ep})

# Carica baseline AdamW (se presente)
baseline_ep = None
if os.path.isdir(f'results/{BASELINE_TAG}'):
    baseline_ep = pd.read_csv(f'results/{BASELINE_TAG}/training_log.csv')

if not loaded:
    print('❌ Nessun risultato Floor. Esegui Cella 3 prima.')
else:
    out_dir = f'opt_plots/floor_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}'
    os.makedirs(out_dir, exist_ok=True)

    # ── Tabella riassuntiva ────────────────────────────────────
    rows = []
    for d in loaded:
        full, ep = d['full'], d['ep']
        best = ep.val_total.min()
        delta = best - BASELINE_VAL
        verdict = '🎯 COLPEVOLE!' if delta < -0.01 else ('≈ comparable' if abs(delta) < 0.005 else '❌ peggio')
        rows.append({
            'tag':       full['tag'].replace('P12_S2D_', ''),
            'override':  ('PINN=0' if full['lambda_phys']==0 else
                          'OU=0' if full['noise_scale']==0 else
                          f'n_train={full["n_train"]}'),
            'best_val':  f'{best:.4f}',
            'vs_baseline': f'{delta:+.4f}',
            'verdict':   verdict,
        })
    df = pd.DataFrame(rows)
    display(Markdown(f'### 📊 Floor Diagnostic — baseline AdamW val={BASELINE_VAL:.4f}'))
    display(df)

    # ── Grafico O1: val_total per epoca (overlay tutti + baseline) ──
    fig, ax = plt.subplots(figsize=(11, 6))
    cmap = plt.cm.tab10(np.linspace(0, 1, len(loaded)))
    for i, d in enumerate(loaded):
        ep = d['ep']
        lbl = d['full']['tag'].replace('P12_S2D_', '')
        ax.plot(ep.epoch, ep.val_total, 'o-', linewidth=2, markersize=7,
                color=cmap[i], label=lbl)
    if baseline_ep is not None:
        ax.plot(baseline_ep.epoch, baseline_ep.val_total, 'k--', linewidth=2.5,
                label=f'baseline AdamW ({BASELINE_VAL:.4f})')
    ax.axhline(BASELINE_VAL, ls=':', color='red', alpha=0.5,
               label=f'floor 2C (0.2805)')
    ax.axhline(0.20, ls=':', color='green', alpha=0.5, label='target val<0.20')
    ax.set_xlabel('epoch'); ax.set_ylabel('val_total')
    ax.set_title('Floor Diagnostic — val_total per epoca')
    ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'{out_dir}/F_val_per_epoch.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── Verdetto ───────────────────────────────────────────────
    display(Markdown('### 🎯 Verdetto'))
    best = min(loaded, key=lambda d: d['ep'].val_total.min())
    best_v = best['ep'].val_total.min()
    delta = best_v - BASELINE_VAL
    if delta < -0.05:
        print(f'🏆 FLOOR SUPERATO: {best["full"]["tag"]} con val={best_v:.4f} ({delta:+.4f})')
        print(f'   Causa floor identificata. Procedere con STEP 2E (mitigation).')
    elif delta < -0.01:
        print(f'✅ MIGLIORAMENTO PARZIALE: {best["full"]["tag"]} val={best_v:.4f} ({delta:+.4f})')
        print(f'   Floor parzialmente spiegato. Potrebbero esserci più cause sovrapposte.')
    else:
        print(f'❌ TUTTI E 3 RESTANO AL FLOOR (best={best_v:.4f}, Δ={delta:+.4f})')
        print(f'   Floor NON è PINN/OU/data. → Po2 quantization (F4 differita).')
    print(f'\n💾 Grafici: {out_dir}/')